# Emotion Lexicons

Prepares the **NRC Emotion Intensity Lexicon** (NRC-EIL) for the keyword decoder, and measures what
it changes before anything is wired in.

Two halves, and they are separate on purpose:

1. **Prepare** — convert the published TSV into the JSON artefact `perception/nrc_eil.py` reads.
   This is **format conversion only**: no axis mapping, no renaming, no thresholding. Every decision
   that changes what the decoder believes lives in `nrc_eil.py`, in version control, beside the
   argument for it. So re-running this notebook is a no-op, which is what makes an untracked
   artefact safe.
2. **Measure** — what the lexicon covers, and the one open question it has to answer before the
   decoder can use it: **does `max` over ten thousand words produce dense vectors where the hand
   table produced sparse ones?** That is a prediction, not a result, until the cells below run.

## Getting the lexicon — a manual step, and it has to be

NRC-EIL is **free for research and teaching but may not be redistributed**, so it is not in this
repository and cannot be fetched automatically — obtaining it means accepting the terms yourself.

1. Go to <https://saifmohammad.com/WebPages/AffectIntensity.htm>
2. Download the lexicon and accept the terms
3. Put `NRC-Emotion-Intensity-Lexicon-v1.txt` in `data_in/`

`data_in/` is already gitignored, so the file cannot be committed by accident. The consequence
worth knowing: a fresh clone cannot build a lexicon decoder until this notebook has been run, and
a run manifest's **content hash** is the only record of what was actually read — a commit cannot
describe a file it does not contain.

In [ ]:
# Paths
#

import json
from collections import Counter
from datetime import UTC, datetime
from pathlib import Path


def repo_root(marker: str = "uv.lock") -> Path:
    """Nearest ancestor of the working directory containing *marker*."""
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / marker).is_file():
            return candidate
    raise FileNotFoundError(f"No {marker} found above {start}")


DATA_IN = repo_root() / "data_in"

SOURCE_TSV = DATA_IN / "NRC-Emotion-Intensity-Lexicon-v1.txt"   # licensed, never committed

# Named for the file it was made from, and dated, because the artefact is untracked and
# nothing else in the repository records either. Carrying the supplier's own stem means the
# pair can be read off a directory listing without opening anything — which is what an
# untracked file most needs. ISO date so the listing sorts, and the stem first so several
# preparations of one lexicon group together once a second lexicon exists.
PREPARED_ON = datetime.now(UTC).date().isoformat()
ARTEFACT = DATA_IN / f"{SOURCE_TSV.stem}.loaded-{PREPARED_ON}.json"

print(f"source   : {SOURCE_TSV}  exists={SOURCE_TSV.exists()}")
print(f"artefact : {ARTEFACT}  exists={ARTEFACT.exists()}")

## Parse the TSV

Three tab-separated columns — `word`, `emotion`, `emotion-intensity-score` — **with or without a
header row, because copies of this file differ.** The one used here starts straight in on
`outraged anger 0.964`, while the version described on the NRC pages carries a header. So the first
line is treated as a header only if its third field will not parse as a number, and the cell
**reports which it found** rather than staying silent — a skip that prints only when it happens
cannot be told apart from a cell that never ran.

Hardcoding "skip line 1" would silently discard a real word from this copy. The check costs two
lines and cannot lose data in either direction.

Only rows with a non-zero score are kept: a word scoring zero on an axis contributes nothing under
the decoder's `max`, and carrying them would multiply the artefact's size for no effect. NRC-EIL
does contain a handful of explicit zeros.

Nothing here knows about `plutchik8/1`. The keys stay the lexicon's own emotion names, including
`joy` — `nrc_eil.PLUTCHIK8_FROM_NRC` is what renames it, and that is a decision, so it lives in
code rather than in this cell.

In [ ]:
# TSV -> {emotion: {word: score}}, in the lexicon's own vocabulary
#

entries: dict[str, dict[str, float]] = {}
skipped_zero, header = 0, None

lines = SOURCE_TSV.read_text(encoding="utf-8").splitlines()

for number, line in enumerate(lines, start=1):
    if not line.strip():
        continue
    try:
        word, emotion, score = line.split("\t")
    except ValueError:
        raise ValueError(f"{SOURCE_TSV.name}:{number}: expected three tab-separated fields, "
                         f"got {line!r}") from None
    try:
        magnitude = float(score)
    except ValueError:
        if number == 1:
            header = line                       # a header, not a defect — copies differ
            continue
        raise ValueError(f"{SOURCE_TSV.name}:{number}: {score!r} is not a number") from None

    if magnitude > 0.0:
        entries.setdefault(emotion, {})[word] = magnitude
    else:
        skipped_zero += 1

# Reported either way. A message printed only in the branch that fired makes "this file has no
# header" indistinguishable from "the cell did not run".
print(f"header: {header!r}" if header else f"header: none — line 1 is data ({lines[0]!r})")
print(f"{len(lines):,} lines -> {len(entries)} emotions, "
      f"{sum(len(v) for v in entries.values()):,} word-emotion pairs "
      f"({skipped_zero:,} zero-scored rows dropped)")
print(f"emotions: {sorted(entries)}")

## Write the artefact

`id` is what reaches every record's `source` field, so a decoder built from this file is
identifiable in `evidence.jsonl` without anyone having to remember which lexicon a run used.
`lexicon`, `prepared_by` and `prepared_on` are for a human who opens the file; the loader ignores
them.

**The filename carries the supplier's stem and the date it was prepared**, so the artefact and the
`.txt` it came from can be paired off a directory listing without opening either — which is what an
untracked file most needs, since neither is described by a commit. The other prepared files here are
snake_case; this one follows its source instead, as the source `.txt` does.

**`id` deliberately carries no date, though the filename does.** Two preparations of the same
source TSV on different days are the *same instrument*, and giving them different `source` strings
would fragment an analysis across dates for no research reason. The filename says which file is on
disk; `id` says which instrument produced a record. A magnitude `floor` is the contrasting case and
*does* join `id`, because a different floor genuinely decodes differently.

Neither the filename nor `prepared_on` is real provenance — both record when this notebook ran, not
what it read. The manifest's **content hash** is what pins that, which is why v0.6.1 made it binding
rather than prudent for exactly this kind of untracked file.

Preparations **accumulate** rather than overwriting, which is the point of dating them: an older
artefact stays loadable, so a run recorded against it can be reproduced. It also means whatever
uses a lexicon must be given an **explicit path** — resolving "the newest matching file" would make
which instrument a run used depend on what happens to be in the directory, and re-running the same
command a week later could then silently decode differently.

There is deliberately **no `representation` field**. The keys above are `joy` and `anticipation` —
the lexicon's vocabulary, not `plutchik8/1`'s, which says `happiness`. Stamping a representation
here would be false on that one axis, and would split "which representation is this lexicon native
to" between an untracked file and `nrc_eil.NATIVE`, leaving two sources of truth that can disagree.

In [ ]:
# Write the prepared artefact, then round-trip it through the real loader
#

from asa.core.representations import EKMAN6, PLUTCHIK8
from asa.perception.nrc_eil import load_tables

prepared = {
    "id": "nrc-eil",                                    # reaches every record's `source`
    "lexicon": SOURCE_TSV.name,                         # for a human reading the file
    "prepared_by": "notebooks/lexicons.ipynb",
    "prepared_on": PREPARED_ON,                         # survives a rename; the filename does not
    "entries": entries,
}

ARTEFACT.write_text(json.dumps(prepared, indent=1, sort_keys=True), encoding="utf-8")
print(f"wrote {ARTEFACT}  ({ARTEFACT.stat().st_size / 1024:.0f} KiB)")

# The artefact is only good if the real loader accepts it, so prove that here rather than
# discovering it at step 7c.
lex = load_tables(ARTEFACT, [PLUTCHIK8, EKMAN6])
print(f"source   : {lex.source}")
for rep_id, table in lex.tables.items():
    print(f"{rep_id:<12} {len(table)} axes, {sum(len(w) for w in table.values()):,} words")

## What it covers, against what it replaces

The hand-written tables hold around forty words each, with magnitudes that are a hand-set ordering
rather than a measurement. The interesting comparison is not just size but **overlap**: how much of
the hand table the lexicon actually contains, and where the two disagree on an axis assignment.

A word the hand table places on one axis and the lexicon on another is worth looking at directly —
it is either a genuine disagreement about the word or a sign the hand table was tuned to the
sentences it was tested on.

In [ ]:
# Coverage and disagreement, ekman6/1
#

from asa.perception.decode_keyword import EKMAN6_KEYWORDS

hand = {axis: set(words) for axis, words in EKMAN6_KEYWORDS.items()}
lexi = {axis: set(words) for axis, words in lex.tables[EKMAN6.id].items()}

print(f"{'axis':<12}{'hand':>6}{'lexicon':>9}{'shared':>8}{'hand-only':>11}")
for axis in EKMAN6.axes:
    was, now = hand.get(axis, set()), lexi.get(axis, set())
    print(f"{axis:<12}{len(was):>6}{len(now):>9}{len(was & now):>8}{len(was - now):>11}")

print()

# A word can honestly sit on SEVERAL axes — the lexicon is full of them — so this compares
# axis *sets* per word. Mapping each word to one axis would silently keep whichever came last
# in iteration order and report ordering as disagreement.
def axes_by_word(table):
    out: dict[str, set[str]] = {}
    for axis, words in table.items():
        for word in words:
            out.setdefault(word, set()).add(str(axis))
    return out


hand_axes, lexi_axes = axes_by_word(hand), axes_by_word(lexi)

disagree = {w: (axes, lexi_axes[w]) for w, axes in hand_axes.items()
            if w in lexi_axes and not (axes & lexi_axes[w])}
absent = sorted(set(hand_axes) - set(lexi_axes))
multi = [w for w, axes in lexi_axes.items() if len(axes) > 1]

print(f"{len(multi):,} of {len(lexi_axes):,} lexicon words sit on more than one ekman6 axis "
      f"({len(multi) / len(lexi_axes):.0%})")
print(f"\n{len(disagree)} hand-table words where the two share NO axis at all:")
for word, (was, now) in sorted(disagree.items()):
    print(f"    {word:<14} hand={sorted(was)}  lexicon={sorted(now)}")
print(f"\n{len(absent)} hand-table words absent from the lexicon: {absent}")

## The density check — the question this notebook exists to answer

The decoder combines matches per axis with **`max`**, chosen for a forty-word table where nought to
two words fire in a sentence. Against ten thousand entries the worry is that *most axes will have
some matching word*, so vectors come out dense and near-uniform where the ground truth is sparse —
which would wreck precision against a binary-labelled corpus without failing anything.

**This is a prediction from the lexicon's size, not a measurement.** The cell below is the
measurement. If the lexicon's median non-zero axis count is close to the number of axes, a
magnitude `floor` is needed and `load_tables(..., floor=…)` is where it goes — and it must reach
the `source` string, because two floors over one lexicon are two instruments.

In [ ]:
# Non-zero axes per row: hand table vs lexicon, over BRIGHTER
#

import asyncio

import pandas as pd

from asa.core.affect import Utterance, utc_now
from asa.perception.decode_keyword import LEXICON_HANDWRITTEN, KeywordDecoder

bench = pd.read_parquet(DATA_IN / "bench_brighter_eng.parquet")
texts = bench["text"].tolist()
print(f"{len(texts):,} BRIGHTER-eng rows")

AT = utc_now()


async def decode_all(decoder, texts):
    return [await decoder.decode(Utterance(text=t, source="input:notebook", at=AT))
            for t in texts]


def live_axes(decoder):
    """How many axes each row lights up, and how many rows fire nothing at all."""
    got = asyncio.run(decode_all(decoder, texts))
    counts = [sum(1 for v in o.affect.values.values() if v > 0.0) for o in got]
    return counts


decoders = {
    "hand": KeywordDecoder(EKMAN6, EKMAN6_KEYWORDS, LEXICON_HANDWRITTEN),
    "nrc": KeywordDecoder(EKMAN6, lex.tables[EKMAN6.id], lex.source),
}

# Ground truth for comparison: how many axes each row is actually labelled with.
truth = [int(r.fillna(0).sum()) for _, r in bench[list(EKMAN6.axes)].iterrows()]

print(f"\n{'':<8}{'0 axes':>9}{'1':>7}{'2':>7}{'3':>7}{'4+':>7}{'median':>9}{'mean':>8}")
for name, decoder in decoders.items():
    counts = live_axes(decoder)                 # one pass — decoding 8.5k rows is not free
    c = Counter(min(n, 4) for n in counts)
    med = sorted(counts)[len(counts) // 2]
    print(f"{name:<8}{c[0]:>9}{c[1]:>7}{c[2]:>7}{c[3]:>7}{c[4]:>7}"
          f"{med:>9}{sum(counts) / len(counts):>8.2f}")

ct = Counter(min(n, 4) for n in truth)
print(f"{'truth':<8}{ct[0]:>9}{ct[1]:>7}{ct[2]:>7}{ct[3]:>7}{ct[4]:>7}"
      f"{sorted(truth)[len(truth) // 2]:>9}{sum(truth) / len(truth):>8.2f}")

### Reading the table

The row that matters is **`nrc` against `truth`**. If `nrc` sits far to the right of `truth` — many
rows lighting three or four axes where the annotators marked one — then `max` over an unfiltered
lexicon is measuring vocabulary rather than affect, and a floor is required before step 7c.

The `hand` row is the other reference point: the 2026-08-09 pilot found 142 of 150 rows decoding to
**nothing at all**, so the hand table's failure is non-matches. If the lexicon's `0 axes` column
collapses while the rest spreads out, the two decoders fail in opposite directions, and that
contrast is itself a result worth writing up.

## What Ekman's six cannot see

`plutchik8/1` exists to be diagnostic, not scored: BRIGHTER carries the `ekman6/1` columns only, so
`anticipation` and `trust` have no ground truth and never will.

What it *can* answer is how often the strongest lexical signal in a real utterance lands on an axis
Ekman's six has no name for. Rows where the plutchik8 decoder fires **only** on those two axes are
the sharpest case: under `ekman6/1` they decode to rest and are indistinguishable from a sentence
carrying no affect at all.

In [ ]:
# Rows whose only lexical signal is on an axis ekman6/1 cannot represent
#

eight = KeywordDecoder(PLUTCHIK8, lex.tables[PLUTCHIK8.id], lex.source)
got8 = asyncio.run(decode_all(eight, texts))

BLIND = ("anticipation", "trust")

only_blind, both, examples = 0, 0, []
for text, o in zip(texts, got8, strict=True):
    live = {str(a) for a, v in o.affect.values.items() if v > 0.0}   # str: axes are StrEnum
    if not live:
        continue
    if live <= set(BLIND):
        only_blind += 1
        if len(examples) < 8:
            examples.append((text[:70], sorted(live)))
    elif live & set(BLIND):
        both += 1

print(f"{only_blind:,} of {len(texts):,} rows ({only_blind / len(texts):.1%}) fire ONLY on "
      f"{' or '.join(BLIND)} — these decode to rest under ekman6/1")
print(f"{both:,} more fire on those axes alongside an ekman6 axis\n")
for text, live in examples:
    print(f"    {live}  {text!r}")

---

## What to carry forward

- **The artefact** is written and round-tripped through `load_tables`. Nothing else reads it yet.
- **The density result** decides whether step 7c passes a `floor`, and what value.
- **The diagnostic number** is a finding for the write-up, not an accuracy figure — it says what a
  six-axis representation cannot see, which is the argument `plutchik8/1` was declared to support.

Nothing here changes what `uv run asa` does. Step 7c is where the lexicon becomes selectable.